# Final U-Net++ talc model

Final training on all 42 manually corrected image/mask pairs after 3-fold validation.

Configuration: `UnetPlusPlus + efficientnet-b3`, `img_size=768`, `batch_size=4`.


In [ ]:
# If needed, run this once on the server kernel.
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install segmentation-models-pytorch albumentations wandb timm


In [ ]:
from pathlib import Path
from datetime import datetime
import random
import subprocess

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import wandb

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('smp:', smp.__version__)


In [ ]:
SEED = 117
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DATA_ROOT = Path('/home/team117/nornik/talc_unetpp_dataset_from_json')
RUNS_DIR = Path('/home/team117/nornik/runs_unetpp_final')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'project': 'nornik-talc-segmentation',
    'run_name': 'final-unetpp-efficientnet-b3-768-all42',
    'data_root': str(DATA_ROOT),
    'model': 'UnetPlusPlus',
    'encoder': 'efficientnet-b3',
    'encoder_weights': 'imagenet',
    'img_size': 768,
    'batch_size': 4,
    'epochs': 25,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'dice_weight': 1.0,
    'bce_weight': 1.0,
    'threshold': 0.5,
    'log_gpu_stats': True,
    'seed': SEED,
}

assert (DATA_ROOT / 'images').exists(), DATA_ROOT
assert (DATA_ROOT / 'masks').exists(), DATA_ROOT

run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
OUT_DIR = RUNS_DIR / f"{run_id}_{CFG['run_name']}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)


In [ ]:
def list_all_pairs():
    img_dir = DATA_ROOT / 'images'
    mask_dir = DATA_ROOT / 'masks'
    image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}])
    pairs = []
    for img_path in image_paths:
        matches = sorted(mask_dir.glob(img_path.stem + '.*'))
        if not matches:
            raise FileNotFoundError(f'No mask for {img_path.name}')
        pairs.append((img_path, matches[0]))
    return pairs

def mask_fraction(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    return float((mask > 127).mean())

pairs = list_all_pairs()
meta = pd.DataFrame({
    'image': [p[0].name for p in pairs],
    'mask': [p[1].name for p in pairs],
    'mask_fraction': [mask_fraction(p[1]) for p in pairs],
})
meta.to_csv(OUT_DIR / 'train_all_manifest.csv', index=False)
print('pairs:', len(pairs))
print(meta['mask_fraction'].describe())
meta.head()


In [ ]:
class TalcSegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype('float32')

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image, mask, img_path.name

train_tfms = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.35),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.25),
    A.Blur(blur_limit=3, p=0.15),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

preview_tfms = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_ds = TalcSegDataset(pairs, train_tfms)
preview_ds = TalcSegDataset(pairs, preview_tfms)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=False)
preview_loader = DataLoader(preview_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True, drop_last=False)


In [ ]:
def denormalize(x):
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(3, 1, 1)
    return torch.clamp(x * std + mean, 0, 1)

bce_loss = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    union = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def total_loss(logits, targets):
    return CFG['bce_weight'] * bce_loss(logits, targets) + CFG['dice_weight'] * dice_loss_from_logits(logits, targets)

def batch_metrics(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    inter = torch.sum(preds * targets, dims)
    pred_sum = torch.sum(preds, dims)
    target_sum = torch.sum(targets, dims)
    union = pred_sum + target_sum - inter
    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)
    return {'dice': dice.mean().item(), 'iou': iou.mean().item()}

def make_wandb_images(images, masks, logits, names, max_items=6):
    images = images.detach().cpu()
    masks = masks.detach().cpu().numpy()
    probs = torch.sigmoid(logits.detach()).cpu().numpy()
    out = []
    for i in range(min(max_items, images.shape[0])):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        gt = (masks[i, 0] > 0.5).astype(np.uint8)
        pred = (probs[i, 0] > CFG['threshold']).astype(np.uint8)
        out.append(wandb.Image(
            (img * 255).astype(np.uint8),
            caption=names[i],
            masks={
                'ground_truth': {'mask_data': gt, 'class_labels': {0: 'background', 1: 'talc'}},
                'prediction': {'mask_data': pred, 'class_labels': {0: 'background', 1: 'talc'}},
            }
        ))
    return out

def get_gpu_stats(device):
    stats = {}
    if device.type != 'cuda' or not CFG.get('log_gpu_stats', True):
        return stats
    stats.update({
        'gpu/torch_mem_allocated_gb': torch.cuda.memory_allocated() / 1024**3,
        'gpu/torch_mem_reserved_gb': torch.cuda.memory_reserved() / 1024**3,
        'gpu/torch_max_mem_allocated_gb': torch.cuda.max_memory_allocated() / 1024**3,
    })
    try:
        query = 'utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,temperature.gpu'
        out = subprocess.check_output(['nvidia-smi', f'--query-gpu={query}', '--format=csv,noheader,nounits'], text=True, timeout=5)
        values = [float(x.strip()) for x in out.strip().splitlines()[0].split(',')]
        stats.update({
            'gpu/utilization_pct': values[0],
            'gpu/memory_utilization_pct': values[1],
            'gpu/memory_used_mb': values[2],
            'gpu/memory_total_mb': values[3],
            'gpu/power_w': values[4],
            'gpu/temperature_c': values[5],
        })
    except Exception as exc:
        stats['gpu/nvidia_smi_error'] = str(exc)[:200]
    return stats


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.UnetPlusPlus(
    encoder_name=CFG['encoder'],
    encoder_weights=CFG['encoder_weights'],
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 0.05)
scaler = GradScaler(device.type, enabled=(device.type == 'cuda'))

print('parameters_m:', round(sum(p.numel() for p in model.parameters()) / 1e6, 2))


In [ ]:
def train_one_epoch():
    model.train()
    losses, dices, ious = [], [], []
    for images, masks, names in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = total_loss(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        m = batch_metrics(logits, masks, CFG['threshold'])
        losses.append(loss.item()); dices.append(m['dice']); ious.append(m['iou'])
    return {'loss': float(np.mean(losses)), 'dice': float(np.mean(dices)), 'iou': float(np.mean(ious))}

@torch.no_grad()
def log_preview():
    model.eval()
    images, masks, names = next(iter(preview_loader))
    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True).float()
    with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
        logits = model(images)
    return make_wandb_images(images, masks, logits, names)


In [ ]:
wandb_run = wandb.init(
    project=CFG['project'],
    name=f"{CFG['run_name']}-{run_id}",
    config=CFG,
    dir=str(OUT_DIR),
    anonymous='allow',
)
wandb.watch(model, log='gradients', log_freq=50)


In [ ]:
history = []

for epoch in range(1, CFG['epochs'] + 1):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    train_m = train_one_epoch()
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    gpu_stats = get_gpu_stats(device)

    row = {
        'epoch': epoch,
        'lr': lr,
        'train_loss': train_m['loss'],
        'train_dice': train_m['dice'],
        'train_iou': train_m['iou'],
        **gpu_stats,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUT_DIR / 'history.csv', index=False)

    log_data = dict(row)
    if epoch == 1 or epoch % 5 == 0 or epoch == CFG['epochs']:
        log_data['train_predictions'] = log_preview()
    wandb.log(log_data, step=epoch)

    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'config': CFG,
    }
    torch.save(ckpt, OUT_DIR / 'last_unetpp_effb3_768_all42.pt')

    print(
        f"Epoch {epoch:03d}/{CFG['epochs']} | "
        f"train loss {train_m['loss']:.4f} dice {train_m['dice']:.4f} iou {train_m['iou']:.4f} | lr {lr:.2e}"
    )

wandb.save(str(OUT_DIR / 'last_unetpp_effb3_768_all42.pt'))
wandb.finish()
print('Checkpoint:', OUT_DIR / 'last_unetpp_effb3_768_all42.pt')


In [ ]:
# Optional local preview from final checkpoint.
ckpt = torch.load(OUT_DIR / 'last_unetpp_effb3_768_all42.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

images, masks, names = next(iter(preview_loader))
images = images.to(device)
with torch.no_grad(), autocast(device_type=device.type, enabled=(device.type == 'cuda')):
    logits = model(images)
probs = torch.sigmoid(logits).detach().cpu().numpy()

fig, axes = plt.subplots(min(4, len(names)), 3, figsize=(12, 4 * min(4, len(names))))
if min(4, len(names)) == 1:
    axes = np.array([axes])
for i in range(min(4, len(names))):
    img = denormalize(images[i].detach().cpu()).permute(1, 2, 0).numpy()
    gt = masks[i, 0].numpy()
    pred = probs[i, 0] > CFG['threshold']
    axes[i, 0].imshow(img); axes[i, 0].set_title(names[i]); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt, cmap='gray'); axes[i, 1].set_title('gt'); axes[i, 1].axis('off')
    axes[i, 2].imshow(img); axes[i, 2].imshow(pred, alpha=0.35, cmap='Blues'); axes[i, 2].set_title('prediction'); axes[i, 2].axis('off')
plt.tight_layout()
